In [5]:
from pathlib import Path

import pandas as pd

## Local

In [6]:
DATA_DIR = Path.cwd().parent / 'data' / 'sample'
# Caso o notebook seja executado a partir da raiz do projeto.
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd() / 'data' / 'sample'

## Colab

## Tratamento e Join PIB

In [7]:
df_pib = (
    pd.read_csv(
        DATA_DIR / 'br_ibge_pib_municipio.csv',
        usecols=['id_municipio', 'ano', 'pib'],
    )
    .query('ano in [2022, 2023]')
)

df_pib.head()

,id_municipio,ano,pib
2216,1100023,2022,3809355000
2217,1302900,2022,649736000
2218,1501402,2022,37604333000
2219,1505700,2022,277171000
2220,1600535,2022,426170000


In [8]:
df_alunos = pd.read_parquet(DATA_DIR / 'df_alunos_sample.parquet')

# Para cada aluno, usa-se o PIB do ano imediatamente anterior.
df_alunos = df_alunos.assign(ano_pib=df_alunos['ano'] - 1)

df_alunos_com_pib = df_alunos.merge(
    df_pib,
    left_on=['id_municipio', 'ano_pib'],
    right_on=['id_municipio', 'ano'],
    how='left',
    validate='m:1',
).drop(columns=['ano_pib', 'ano_y']).rename(columns={'ano_x': 'ano'})

df_alunos_com_pib.head()

,id_municipio,id_escola,id_aluno,codigo_uf,sigla_uf,regiao,serie,rede,caderno,presenca,...,meta_brasil_2025_pct,meta_brasil_2026_pct,meta_brasil_2027_pct,meta_brasil_2028_pct,meta_brasil_2029_pct,meta_brasil_2030_pct,meta_brasil_percentual_participacao_pct,ano,rede_desc,pib
0,4120903,60033120,41000620,41,PR,Sul,2,3,10,1,...,63.77,67.47,70.97,74.23,77.24,80,87.37,2024,Municipal,1927945000
1,1101401,60000230,11001863,11,RO,Norte,2,3,12,1,...,63.77,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,452939000
2,2927408,60017747,29117397,29,BA,Nordeste,2,3,8,1,...,63.77,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,68764218000
3,1300904,60001148,13005111,13,AM,Norte,2,3,1,1,...,63.77,67.47,70.97,74.23,77.24,80,87.37,2024,Municipal,208247000
4,3303500,60024793,33066826,33,RJ,Sudeste,2,3,13,1,...,63.77,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,19911894000


In [9]:
municipios_com_pib = df_alunos_com_pib.loc[
    df_alunos_com_pib['pib'].notna(), 'id_municipio'
].nunique()
municipios_sem_pib = df_alunos_com_pib.loc[
    df_alunos_com_pib['pib'].isna(), 'id_municipio'
].nunique()

pd.DataFrame(
    {
        'situação': ['Com PIB', 'Sem PIB'],
        'municípios distintos': [municipios_com_pib, municipios_sem_pib],
    }
)

,situação,municípios distintos
0,Com PIB,5539
1,Sem PIB,0


## Tratamento e Join População

In [12]:
# A fonte contém uma linha por município; agregamos para obter a população por UF e ano.
df_populacao = (
    pd.read_csv(
        DATA_DIR / 'br_ibge_populacao_municipio.csv',
        usecols=['ano', 'id_municipio', 'populacao'],
    )
    .query('ano in [2023, 2024]')
    .groupby(['ano', 'id_municipio'], as_index=False)['populacao'].sum()
)

In [13]:
df_alunos_com_pib_populacao = df_alunos_com_pib.merge(
    df_populacao,
    on=['id_municipio', 'ano'],
    how='left',
    validate='m:1',
)

pd.DataFrame(
    {
        'situação': ['Com população', 'Sem população'],
        'quantidade de registros': [
            df_alunos_com_pib_populacao['populacao'].notna().sum(),
            df_alunos_com_pib_populacao['populacao'].isna().sum(),
        ],
    }
)

,situação,quantidade de registros
0,Com população,386816
1,Sem população,0


In [14]:
df_alunos_com_pib_populacao

,id_municipio,id_escola,id_aluno,codigo_uf,sigla_uf,regiao,serie,rede,caderno,presenca,...,meta_brasil_2026_pct,meta_brasil_2027_pct,meta_brasil_2028_pct,meta_brasil_2029_pct,meta_brasil_2030_pct,meta_brasil_percentual_participacao_pct,ano,rede_desc,pib,populacao
0,4120903,60033120,41000620,41,PR,Sul,2,3,10,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Municipal,1927945000,31405.0
1,1101401,60000230,11001863,11,RO,Norte,2,3,12,1,...,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,452939000,11548.0
2,2927408,60017747,29117397,29,BA,Nordeste,2,3,8,1,...,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,68764218000,2417678.0
3,1300904,60001148,13005111,13,AM,Norte,2,3,1,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Municipal,208247000,17885.0
4,3303500,60024793,33066826,33,RJ,Sudeste,2,3,13,1,...,67.47,70.97,74.23,77.24,80,86.00,2023,Municipal,19911894000,785867.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386811,1709500,60004247,17006121,17,TO,Norte,2,4,4,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Privada,3695017000,89574.0
386812,5221601,60040930,52075072,52,GO,Centro-Oeste,2,2,4,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Estadual,1637352000,44150.0
386813,1709500,60004247,17006136,17,TO,Norte,2,4,5,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Privada,3695017000,89574.0
386814,1709500,60004247,17006114,17,TO,Norte,2,4,4,1,...,67.47,70.97,74.23,77.24,80,87.37,2024,Privada,3695017000,89574.0


## Tratamento e Join Gestor

In [29]:
df_gestor = pd.read_csv(
    DATA_DIR / 'br_ibge_munic_indicadores_perfil_gestor.csv',
)

maior_ano_gestor = df_gestor['ano'].max()
df_gestor = df_gestor.query('ano == @maior_ano_gestor & tema == "eduacao"')

print(f'Maior ano disponível: {maior_ano_gestor}')
df_gestor.head()

Maior ano disponível: 2021


,ano,sigla_uf,id_municipio,tema,recusa,sem_gestor,caracterizacao_orgao_gestor,sexo,idade,cor_raca,cor_raca_autodeclarado,escolaridade,escolaridade_formacao
1388,2021,PA,1505908,eduacao,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1394,2021,SC,4216602,eduacao,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1397,2021,PE,2604205,eduacao,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17490,2021,RO,1100023,eduacao,0,0,Secretaria exclusiva,Feminino,56.0,Parda,1.0,Especialização,Pedagogia
17493,2021,RO,1100106,eduacao,0,0,Secretaria exclusiva,Feminino,50.0,Parda,0.0,Especialização,Pedagogia


In [30]:
# Tirando variáveis sensíveis
df_gestor = (
    df_gestor
    .drop(columns=['ano', 'tema', 'sexo', 'cor_raca', 'idade', 'cor_raca_autodeclarado', 'sigla_uf'])
)

df_gestor

,id_municipio,recusa,sem_gestor,caracterizacao_orgao_gestor,escolaridade,escolaridade_formacao
1388,1505908,1,0,NaN,NaN,NaN
1394,4216602,1,0,NaN,NaN,NaN
1397,2604205,0,0,NaN,NaN,NaN
17490,1100023,0,0,Secretaria exclusiva,Especialização,Pedagogia
17493,1100106,0,0,Secretaria exclusiva,Especialização,Pedagogia
...,...,...,...,...,...,...
76140,5213509,0,0,Secretaria em conjunto com outras políticas se...,Ensino superior completo,Pedagogia
76143,5214903,0,0,Secretaria em conjunto com outras políticas se...,Especialização,Pedagogia
76148,5215603,0,0,Secretaria em conjunto com outras políticas se...,Especialização,Pedagogia
76152,5218300,0,0,Secretaria em conjunto com outras políticas se...,Especialização,História


In [32]:
df_alunos_com_pib_populacao_gestor = df_alunos_com_pib_populacao.merge(
    df_gestor,
    on=['id_municipio'],
    how='left',
    validate='m:1',
)

pd.DataFrame(
    {
        'situação': ['Com info gestor', 'Sem info gestor'],
        'quantidade de registros': [
            df_alunos_com_pib_populacao_gestor['recusa'].notna().sum(),
            df_alunos_com_pib_populacao_gestor['recusa'].isna().sum(),
        ],
    }
)

,situação,quantidade de registros
0,Com info gestor,386816
1,Sem info gestor,0


In [33]:
df_alunos_com_pib_populacao_gestor

,id_municipio,id_escola,id_aluno,codigo_uf,sigla_uf,regiao,serie,rede,caderno,presenca,...,meta_brasil_percentual_participacao_pct,ano,rede_desc,pib,populacao,recusa,sem_gestor,caracterizacao_orgao_gestor,escolaridade,escolaridade_formacao
0,4120903,60033120,41000620,41,PR,Sul,2,3,10,1,...,87.37,2024,Municipal,1927945000,31405.0,0,0,Secretaria exclusiva,Especialização,Administração
1,1101401,60000230,11001863,11,RO,Norte,2,3,12,1,...,86.00,2023,Municipal,452939000,11548.0,0,0,Secretaria exclusiva,Mestrado,Letras
2,2927408,60017747,29117397,29,BA,Nordeste,2,3,8,1,...,86.00,2023,Municipal,68764218000,2417678.0,0,0,Secretaria exclusiva,Mestrado,Outra
3,1300904,60001148,13005111,13,AM,Norte,2,3,1,1,...,87.37,2024,Municipal,208247000,17885.0,0,0,Secretaria em conjunto com outras políticas se...,Ensino superior completo,História
4,3303500,60024793,33066826,33,RJ,Sudeste,2,3,13,1,...,86.00,2023,Municipal,19911894000,785867.0,0,0,Secretaria exclusiva,Especialização,Matemática
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386811,1709500,60004247,17006121,17,TO,Norte,2,4,4,1,...,87.37,2024,Privada,3695017000,89574.0,0,0,Secretaria exclusiva,Mestrado,Pedagogia
386812,5221601,60040930,52075072,52,GO,Centro-Oeste,2,2,4,1,...,87.37,2024,Estadual,1637352000,44150.0,0,0,Secretaria exclusiva,Especialização,História
386813,1709500,60004247,17006136,17,TO,Norte,2,4,5,1,...,87.37,2024,Privada,3695017000,89574.0,0,0,Secretaria exclusiva,Mestrado,Pedagogia
386814,1709500,60004247,17006114,17,TO,Norte,2,4,4,1,...,87.37,2024,Privada,3695017000,89574.0,0,0,Secretaria exclusiva,Mestrado,Pedagogia


## Save base final

In [34]:
OUTPUT_PATH = DATA_DIR / 'data_sample_modeling.parquet'

df_alunos_com_pib_populacao_gestor.to_parquet(OUTPUT_PATH, index=False)
print(f'Base salva em: {OUTPUT_PATH}')

Base salva em: c:\Users\Shoiti\Desktop\Shoiti\PosTech - AI Scientist\tech-challenge-3\data\sample\data_sample_modeling.parquet
